# Buddy-Up ML Training on Kaggle

One notebook to **authenticate Kaggle → pull `buddy-up-data` → train → export
(ONNX INT8 + TFLite + model card) → publish artifacts** back to Kaggle — the
Kaggle twin of the local notebooks in this folder, using the same shared
bootstrap (`../training/bootstrap.py`) and export helpers (`../training/tf_utils.py`).

**Model**: `toxicity_classifier` (text moderation for `app/moderation_engine.py`)
is implemented end-to-end here as the reference pipeline. The other models
(`food_classifier`, `nsfw_classifier`, `matching_embeddings`, `feed_ranker`,
`workout_forecast`, …) run from their dedicated notebooks in this folder with
cells 2–4 of this notebook prepended for auth + data.

**How to run**
1. kaggle.com → Code → New Notebook → enable GPU → File ▸ Import Notebook → this file.
2. *(recommended)* Add-ons ▸ Secrets ▸ add a secret named `KAGGLE_API_TOKEN`
   with your `KGAT_…` token — the auth cell picks it up automatically.
3. Attach the `buddy-up-data` dataset (Add Input) **or** let cell 3 pull it via the API.
4. Run all. Artifacts land in `/kaggle/working/buddyup-output` and are published
   to the `<you>/buddyup-models` Kaggle dataset in the final cell.

> ⚠️ **Token hygiene** — a `KGAT_…` token is a credential. It lives in the
> gitignored repo `.env` (`KAGGLE_API_TOKEN=…`) or a Kaggle Secret — never in
> committed code. If it was ever committed to a public repo, **rotate it**
> (kaggle.com → Settings → API → Create New Token).


In [ ]:
import importlib.util
import os
import pathlib
import sys

ON_KAGGLE = pathlib.Path('/kaggle').is_dir()
WORK = pathlib.Path('/kaggle/working') if ON_KAGGLE else pathlib.Path.cwd()
WORK.mkdir(parents=True, exist_ok=True)

# ── 1. locate the Buddy-Up ai_service (training/ + data/ + notebooks/) ────────
ai = None
p = pathlib.Path(os.getcwd()).resolve()
while p != p.parent and ai is None:                      # local / VS Code walk-up
    for cand in (p / 'backend' / 'ai_service', p / 'ai_service', p):
        if (cand / 'training').is_dir() and (cand / 'data').is_dir() and (cand / 'notebooks').is_dir():
            ai = cand
            break
    p = p.parent

if ai is None and ON_KAGGLE:                             # attached repo/utility dataset
    for cand in sorted(pathlib.Path('/kaggle/input').glob('*')):
        if (cand / 'training').is_dir() and (cand / 'notebooks').is_dir():
            ai = cand
            break

if ai is None:                                           # last resort: clone the repo
    repo = WORK / 'Buddy-Up'
    if not (repo / 'backend' / 'ai_service' / 'training').is_dir():
        import subprocess
        subprocess.run(
            ['git', 'clone', '--depth', '1',
             'https://github.com/Bud-Social/Buddy-Up.git', str(repo)],
            check=True,
        )
    ai = repo / 'backend' / 'ai_service'

AI = pathlib.Path(ai).resolve()
sys.path.insert(0, str(AI / 'training'))
sys.path.insert(0, str(AI))                              # training.* namespace package
# /kaggle/input is read-only — notebooks/ cwd only when it is writable
try:
    (AI / 'notebooks').mkdir(parents=True, exist_ok=True)
    os.chdir(AI / 'notebooks')                           # legacy ../data ../models paths
except OSError:
    os.chdir(WORK)

# ── load the repo .env (BUDDY_SCALE, KAGGLE_API_TOKEN, …) BEFORE bootstrap ──
# bootstrap reads BUDDY_SCALE at import time, so this must run first. Uses
# python-dotenv when available, else a tiny built-in parser (Kaggle-safe).
def _find_dotenv(start):
    p = pathlib.Path(start).resolve()
    while p != p.parent:
        f = p / '.env'
        if f.is_file():
            return f
        p = p.parent
    return None

_env_file = _find_dotenv(AI)
try:
    from dotenv import load_dotenv
    load_dotenv(_env_file)
except ImportError:
    if _env_file:
        for _line in _env_file.read_text().splitlines():
            _line = _line.strip()
            if not _line or _line.startswith('#') or '=' not in _line:
                continue
            _k, _, _v = _line.partition('=')
            os.environ.setdefault(_k.strip(), _v.strip().strip('"').strip("'"))
print('[env]', _env_file or 'no .env found', '| BUDDY_SCALE =', os.environ.get('BUDDY_SCALE'),
      '| KAGGLE_API_TOKEN =', 'set' if os.environ.get('KAGGLE_API_TOKEN') else 'missing')

print('ai_service:', AI)

# ── 2. guarded installs (no-op when already present, e.g. Kaggle TF image) ────
_missing = [m for m in ['tensorflow', 'tf2onnx', 'onnxruntime', 'kagglehub']
            if importlib.util.find_spec(m) is None]
if _missing:
    %pip install -q {" ".join(_missing)}

# ── 3. import the shared bootstrap (init() runs after the data root is set) ──
try:
    import training.bootstrap as bootstrap               # noqa: F401
    from training.bootstrap import DATA_ROOT, OUTPUT_ROOT, MODEL_ROOT  # noqa: F401
    print('[bootstrap] module loaded — init() deferred to the data cell')
except Exception as _boot_err:                           # upgrade, never brick
    print('[bootstrap] unavailable:', repr(_boot_err))
    bootstrap = None

from tf_utils import on_gpu, set_memory_growth, tf_version  # noqa: E402
print('tf', tf_version(), '| gpu:', on_gpu())
if on_gpu():
    set_memory_growth()


In [ ]:
# ── Kaggle authentication (KGAT_ API token) ───────────────────────────────────
# Resolution order:
#   1. Kaggle notebook Secret `KAGGLE_API_TOKEN`  (recommended — never in code)
#   2. environment variable `KAGGLE_API_TOKEN`
#   3. repo `.env` → KAGGLE_API_TOKEN=KGAT_… (gitignored; the single place the
#      token lives outside Kaggle)
import os

def _resolve_kaggle_token() -> str:
    # 1) Kaggle notebook secrets
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret('KAGGLE_API_TOKEN')
        if tok:
            print('[auth] token from Kaggle Secrets')
            return tok
    except Exception:
        pass
    # 2) environment
    tok = os.environ.get('KAGGLE_API_TOKEN', '')
    if tok:
        print('[auth] token from environment')
        return tok
    # 3) repo .env (walk up from the ai_service dir; keeps the token out of code)
    p = pathlib.Path(AI).resolve()
    while p != p.parent:
        f = p / '.env'
        if f.is_file():
            for line in f.read_text().splitlines():
                line = line.strip()
                if line and not line.startswith('#') and '=' in line:
                    k, _, v = line.partition('=')
                    os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
            tok = os.environ.get('KAGGLE_API_TOKEN', '')
            if tok:
                print('[auth] token from repo .env:', f)
                return tok
            print('[auth] no KAGGLE_API_TOKEN in', f)
        p = p.parent
    return ''

KAGGLE_TOKEN = _resolve_kaggle_token()
assert KAGGLE_TOKEN.startswith(('KGAT_', 'kaggle_')) or len(KAGGLE_TOKEN) >= 20, (
    'Kaggle token looks invalid — expected a KGAT_… API token')
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_TOKEN

# Legacy kaggle.json (some CLI paths still expect it). We resolve the username
# through the API; only written when we can determine it.
import json as _json
from pathlib import Path as _Path

KAGGLE_USERNAME = os.environ.get('KAGGLE_USERNAME', '')
_kaggle_dir = _Path.home() / '.kaggle'
_kaggle_dir.mkdir(parents=True, exist_ok=True)

import kagglehub  # noqa: E402
try:
    me = kagglehub.whoami()          # verifies the token + returns the identity
    KAGGLE_USERNAME = (me or {}).get('username', KAGGLE_USERNAME) if isinstance(me, dict) else KAGGLE_USERNAME
    print('[auth] whoami:', me)
except Exception as e:               # token invalid or kagglehub too old
    print('[auth] whoami failed:', repr(e))

if KAGGLE_USERNAME and not (_kaggle_dir / 'kaggle.json').exists():
    # KGAT tokens are opaque; kaggle.json needs username+key, so write the token
    # into the new-style credential file the newer tooling also accepts.
    (_kaggle_dir / 'kaggle.json').write_text(_json.dumps({
        'username': KAGGLE_USERNAME, 'key': KAGGLE_TOKEN,
    }))
try:
    os.chmod(_kaggle_dir / 'kaggle.json', 0o600)
except Exception:
    pass

print('KAGGLE_USERNAME =', KAGGLE_USERNAME or '(unknown — uploads will need it)')
assert KAGGLE_USERNAME, 'Could not resolve the Kaggle username from the token'


In [ ]:
# ── Data: pull `buddy-up-data` (or use the attached input), then bootstrap ────
# The repo's DVC-tracked corpora (Reddit IRL comments, profanity lexicon, Gen-Z
# slang, Food.com, …) ship as the Kaggle dataset `<you>/buddy-up-data`. When the
# dataset is attached as a notebook input, /kaggle/input/buddy-up-data already
# has it — otherwise cell 3 of this notebook pulls it via the API token.
import os
import pathlib

data_root = None
attached = pathlib.Path('/kaggle/input/buddy-up-data')
if attached.is_dir():
    data_root = attached
    print('[data] using attached input:', data_root)
else:
    try:
        import kagglehub
        data_root = pathlib.Path(kagglehub.dataset_download(f'{KAGGLE_USERNAME}/buddy-up-data'))
        print('[data] downloaded via API:', data_root)
    except Exception as e:
        print('[data] buddy-up-data pull failed ('
              f'{type(e).__name__}: {e}) — falling back to the repo data/ dir; '
              'synthetic bootstrap data will be used where corpora are missing.')
        data_root = AI / 'data'

if data_root is not None and pathlib.Path(data_root).is_dir():
    os.environ['BUDDY_DATA_ROOT'] = str(data_root)

# Resolve scale (smoke | demo | full) BEFORE init so paths/threads/seeds land right.
# pass the .env scale explicitly — a bootstrap module cached by an
# earlier run in this kernel would otherwise report a stale scale
CFG = (bootstrap.init(scale=os.environ.get('BUDDY_SCALE') or None)
       if bootstrap else {})
DATA_ROOT_P = pathlib.Path(CFG.get('data_root', os.environ.get('BUDDY_DATA_ROOT', str(AI / 'data'))))
OUTPUT_ROOT_P = pathlib.Path(CFG.get('output_root', str(WORK / 'buddyup-output')))
MODEL_ROOT_P = pathlib.Path(CFG.get('model_root', str(OUTPUT_ROOT_P / 'models')))
print('env        :', CFG.get('env', '?'))
print('scale      :', CFG.get('scale', '?'))
print('data_root  :', DATA_ROOT_P)
print('model_root :', MODEL_ROOT_P)

irl_csv = DATA_ROOT_P / 'the-reddit-irl-dataset-comments.csv'
print('IRL comments CSV present:', irl_csv.is_file(), '→', irl_csv)


In [ ]:
# ── Run configuration ─────────────────────────────────────────────────────────
MODEL = 'toxicity_classifier'   # this notebook's reference pipeline
VERSION = '1.0.0'               # bump per retrain; used in artifact + card names
SCALE = CFG.get('scale', 'demo')  # smoke | demo | full  (override: BUDDY_SCALE env)

USE_SLANG = os.environ.get('BUDDY_SLANG', '1') == '1'
EPOCHS = {'smoke': 1, 'demo': 3, 'full': 6}[SCALE]
N_SAMPLES = {'smoke': 4_000, 'demo': 80_000, 'full': 500_000}[SCALE]

print(f'MODEL={MODEL} VERSION={VERSION} SCALE={SCALE} EPOCHS={EPOCHS} N={N_SAMPLES:,}')


In [ ]:
# ── Dataset: Reddit IRL bodies + lexicon/slang bootstrapped labels ────────────
# Mirrors moderation_text.ipynb. If the 2.3 GB IRL CSV is unavailable on Kaggle
# (repo data/ is DVC-tracked and not in the git clone), fall back to a
# lexicon-driven synthetic corpus so the pipeline stays runnable end-to-end.
import numpy as np
import pandas as pd
from buddy_data import profanity, genz_slang

prof = profanity()
slang = genz_slang()
bad = set()
for col in ('canonical_form_1', 'canonical_form_2', 'canonical_form_3'):
    if col in prof.columns:
        bad |= {w.lower() for w in prof[col].dropna().astype(str)}
hard_slang = set()
if USE_SLANG and {'slang_term', 'sentiment_score', 'intensity_score'} <= set(slang.columns):
    hard_slang = set(
        slang.loc[(slang['sentiment_score'] <= -0.2) | (slang['intensity_score'] >= 0.8),
                  'slang_term'].dropna().str.lower()
    )

irl_csv = DATA_ROOT_P / 'the-reddit-irl-dataset-comments.csv'
if irl_csv.is_file():
    rng = np.random.default_rng(42)
    chunks = pd.read_csv(irl_csv, usecols=['body'], chunksize=250_000, low_memory=False)
    texts, want = [], N
    for ci, chunk in enumerate(chunks):
        col = chunk['body'].dropna().astype(str)
        texts.append(col.sample(n=min(want, len(col)), random_state=42 + ci))
        want -= len(texts[-1])
        if want <= 0:
            break
    texts = pd.concat(texts).tolist()[:N]
    src = 'irl'
else:
    # Synthetic clean/toxic mix around the lexicon seeds.
    rng = np.random.default_rng(42)
    clean_seeds = ['i love this recipe, thanks!', 'great post, keep it up!',
                   'leg day was brutal today hahaha', 'anyone tried this protein?',
                   'hahaha lol that is amazing', 'rest day and stretching for me',
                   'new personal best on squats!', 'what a beautiful morning run']
    toxic_seeds = sorted(bad)[:400] or ['stupid', 'idiot', 'trash', 'garbage human']
    slang_seeds = sorted(hard_slang)[:400]
    texts = []
    while len(texts) < N:
        r = rng.random()
        if r < 0.60:
            texts.append(str(rng.choice(clean_seeds)))
        elif r < 0.85 and toxic_seeds:
            w = str(rng.choice(toxic_seeds))
            texts.append(f'{rng.choice(clean_seeds)[:12]} {w}')
        elif slang_seeds:
            texts.append(f'{rng.choice(clean_seeds)[:12]} {rng.choice(slang_seeds)}')
        else:
            texts.append(str(rng.choice(clean_seeds)))
    texts = texts[:N]
    src = 'synthetic'

def to_label(s: str) -> float:
    s = s.lower()
    if any(w in s for w in bad):
        return 1.0
    if USE_SLANG and any(w in s for w in hard_slang):
        return 0.8
    return 0.0

df = pd.DataFrame({'text': texts})
df['label'] = df['text'].map(to_label)
print(f'source={src} | n={len(df):,} | label distribution:')
print(df['label'].value_counts(normalize=True).round(3))


In [ ]:
# ── Split + vectorizer + BiLSTM classifier (TF/Keras) ─────────────────────────
from sklearn.model_selection import train_test_split
import tensorflow as tf

train, val = train_test_split(df, test_size=0.2, random_state=42,
                              stratify=(df['label'] > 0).astype(int)
                              if (df['label'] > 0).any() else None)
if (train['label'] > 0).sum() < 50:              # keep positives learnable
    pos = df[df['label'] > 0]
    if len(pos) >= 20:
        extra = pos.sample(n=min(len(pos), 2000), random_state=7,
                           replace=len(pos) < 2000)
        train = pd.concat([train, extra])

MAX_TOKENS, SEQ, EMB, HID = 40_000, 128, 96, 48
vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS, output_sequence_length=SEQ,
    standardize='lower_and_strip_punctuation')
vectorizer.adapt(np.array(train['text']))
print('vocab size:', vectorizer.vocabulary_size(), '| train rows:', len(train))

inp = tf.keras.Input(shape=(SEQ,), dtype='int64')
x = tf.keras.layers.Embedding(MAX_TOKENS + 2, EMB)(inp)
x = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(HID, return_sequences=True))(x)
x = tf.keras.layers.GlobalAveragePooling1D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
out = tf.keras.layers.Dense(1, activation='sigmoid')(x)
model = tf.keras.Model(inp, out)
model.compile(tf.keras.optimizers.Adam(1e-3), 'binary_crossentropy', metrics=['accuracy'])
model.summary()


In [ ]:
# ── Train (tokenize once, batch on CPU/GPU) ───────────────────────────────────
import time

Xtr = vectorizer(np.array(train['text'])).numpy()
Ytr = train['label'].to_numpy()
Xv = vectorizer(np.array(val['text'])).numpy()
Yv = val['label'].to_numpy()

t0 = time.time()
hist = model.fit(Xtr, Ytr, epochs=EPOCHS, batch_size=256,
                 validation_data=(Xv, Yv), verbose=1)
print(f'train {time.time() - t0:.0f}s')


In [ ]:
# ── Evaluate: AUC / AP + F1-optimal threshold + sanity probe ──────────────────
from sklearn.metrics import (average_precision_score, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score)

Yb = (Yv > 0).astype(int)
p = model.predict(Xv, batch_size=512)[:, 0]
if Yb.sum() == 0 or len(np.unique(Yb)) < 2:
    print('WARNING: val split has no positives — raise SCALE for real signal.')
    auc = ap = float('nan')
    best = 0.5
else:
    auc = roc_auc_score(Yb, p)
    ap = average_precision_score(Yb, p)
    prec, rec, thr = precision_recall_curve(Yb, p)
    f1 = 2 * prec * rec / (prec + rec + 1e-9)
    best = float(thr[np.argmax(f1[:-1])])
    print(f'val AUC={auc:.3f} AP={ap:.3f}')
    print(f'@0.5      P={precision_score(Yb, (p > 0.5).astype(int)):.3f} '
          f'R={recall_score(Yb, (p > 0.5).astype(int)):.3f} '
          f'F1={f1_score(Yb, (p > 0.5).astype(int)):.3f}')
    print(f'@F1-best  threshold={best:.3f} '
          f'F1={f1_score(Yb, (p > best).astype(int)):.3f}')

probe = pd.DataFrame({'text': ['i love this recipe, thanks!', 'shut up you stupid idiot',
                               'great post!', 'this is the worst thing ive ever seen',
                               'hahaha lol']})
Xp = vectorizer(np.array(probe['text'])).numpy()
probe['score'] = model.predict(Xp)[:, 0].round(3)
print(probe.to_string(index=False))


In [ ]:
# ── Export: ONNX (+ dynamic INT8) + vectorizer vocab + model card + run meta ──
# Serving contract: app/ml/serving.py::load_preferred('toxicity_classifier')
# loads the *_int8.onnx artifact; the promotion gate accepts run-metadata.json
# next to the artifact as the metric source.
import json
from pathlib import Path
from tf_utils import export_keras_onnx, quantize_dynamic_onnx, mlflow_log

out_dir = Path(MODEL_ROOT_P)
onnx_path = export_keras_onnx(
    model, out_dir, MODEL, VERSION,
    input_signature=[tf.TensorSpec((None, SEQ), tf.int64, name='input_ids')])
int8_path = quantize_dynamic_onnx(onnx_path)

vocab = {'max_tokens': MAX_TOKENS, 'sequence_length': SEQ,
         'vocabulary': vectorizer.get_vocabulary(), 'threshold': best}
(out_dir / f'{MODEL}_vectorizer.json').write_text(json.dumps(vocab))

card = {
    'name': MODEL, 'version': VERSION,
    'task': 'binary text toxicity classification (toxic/clean)',
    'algorithm': 'TextVectorization + BiLSTM (TF/Keras)',
    'training_data': 'Reddit r/IRL comments + profanity lexicon + Gen-Z slang prior',
    'metrics': {'val_auc': float(auc), 'val_ap': float(ap), 'threshold': float(best)},
    'limitations': 'lexicon-bootstrapped labels (weak supervision); swap in '
                   'Jigsaw/HF toxicity as gold labels for production training',
    'artifact_path': str(int8_path),
}
(out_dir / f'{MODEL}-{VERSION}.card.json').write_text(json.dumps(card, indent=2))

meta = {'name': MODEL, 'version': VERSION, 'artifact_path': str(int8_path),
        'framework': 'tensorflow',
        'metrics': {'val_auc': float(auc), 'val_ap': float(ap), 'threshold': float(best)}}
mlflow_log(meta)
if bootstrap:
    bootstrap.save_run_metadata({'model': MODEL, 'version': VERSION,
                                 'metrics': meta['metrics']})
print('exported:', int8_path)
print('output tree:')
for f in sorted(Path(OUTPUT_ROOT_P).rglob('*')):
    if f.is_file():
        print('  ', f.relative_to(OUTPUT_ROOT_P), f'({f.stat().st_size / 1e6:.2f} MB)')


In [ ]:
# ── Publish: push artifacts to the `<you>/buddyup-models` Kaggle dataset ──────
# Creates the dataset on first run, versions it afterwards. The AI-service CI /
# model-promotion gate pulls new versions from here.
import json
import shutil

publish_dir = WORK / 'buddyup-models'
if publish_dir.exists():
    shutil.rmtree(publish_dir)
publish_dir.mkdir(parents=True)
for f in sorted(pathlib.Path(OUTPUT_ROOT_P).rglob('*')):
    if f.is_file() and f.stat().st_size < 500 * 1024 * 1024:   # Kaggle per-file cap
        rel = f.relative_to(OUTPUT_ROOT_P)
        dest = publish_dir / MODEL / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, dest)

nl = chr(10)
readme.write_text(nl.join([
    '# Buddy-Up trained models', '',
    f'- model: {MODEL}', f'- version: {VERSION}', f'- scale: {SCALE}',
    f'- metrics: {metrics_json}',
    '- exported: ONNX (fp32 + dynamic INT8), vectorizer vocab, model card, run metadata',
]))

handle = f'{KAGGLE_USERNAME}/buddyup-models'
try:
    import kagglehub
    url = kagglehub.dataset_upload(
        handle, str(publish_dir),
        version_notes=f'{MODEL} {VERSION} scale={SCALE} '
                      f'auc={auc:.4f} ap={ap:.4f}')
    print('published:', url)
except Exception as e:
    print('kagglehub upload failed:', repr(e))
    print('Fallback: set KAGGLE_API_TOKEN (or KAGGLE_USERNAME/KAGGLE_KEY from '
          '~/.kaggle/kaggle.json), install the kaggle CLI, then run —')
    print(f'  kaggle datasets init -p {publish_dir}')
    print(f'  kaggle datasets create -p {publish_dir} --dir-mode zip')


## Next steps

1. **Promotion gate** — the model-ci job accepts a `run-metadata.json` next to
   the artifact (written by the export cell) as its metric source.
2. **Serving** — the AI service lazy-loads the `_int8.onnx` artifact via
   `app/ml/serving.py::load_preferred('toxicity_classifier')` from
   `AI_MODEL_CACHE_DIR`; sync the model card into the Django `ModelMetadata`
   table via the `apps.ai` sync endpoint.
3. **Other models** — prepend cells 1–3 (bootstrap + auth + data) to any
   dedicated notebook in this folder (`food_recognition`, `moderation_image`,
   `matching_embeddings`, `feed_ranking`, `workout_time_series`, …) to run them
   on Kaggle with the same token + dataset plumbing.
4. **Security** — if the `KGAT_…` token was ever pasted into code that gets
   committed, rotate it at kaggle.com → Settings → API and switch to the
   notebook-Secrets path in the auth cell.
